[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/089.%20The%20Single%20vs.%20Dual%20Sourcing%20Problem/P89-Tier-2_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 89. The Single vs. Dual Sourcing Problem

## Tier 2: The Classic Heuristic (Priority-Based Allocation Algorithm)

### Key assumptions
- Components can be ranked by criticality scores
- Suppliers can be evaluated using composite scoring functions
- Priority-based sequential decision making
- Threshold-based dual sourcing decisions
- Weighted criteria: cost efficiency, supply risk, and operational complexity

### Approach (step-by-step)
1. **Component Ranking**: Sort components by criticality scores
2. **Supplier Scoring**: Calculate composite scores for each supplier-component pair
3. **Threshold Evaluation**: Apply dual sourcing thresholds based on score separations
4. **Sequential Allocation**: Make sourcing decisions component by component
5. **Confidence Assessment**: Evaluate decision confidence based on score gaps

### What to look for in the results
- Fast computation time for large problem instances
- Clear decision rules based on score thresholds
- Component-specific sourcing strategies
- Confidence metrics for decision quality
- Comparison with mathematical optimization results

### Concrete example (from the source)
We'll implement the priority-based algorithm with:
- Component criticality scores: Component 1 = 0.9, Component 2 = 0.6
- Annual demands: Component 1 = 460 units, Component 2 = 810 units
- Weighted scoring function with weights (0.4, 0.4, 0.2) for cost, reliability, and capacity
- Threshold-based dual sourcing decisions

In [1]:
from typing import Tuple, List, Dict, Set

# Import required libraries for heuristic implementation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional
import heapq
import time
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
@dataclass
class Component:
    """Component data structure with criticality and demand information"""
    name: str
    criticality_score: float  # 0-1 scale, higher = more critical
    annual_demand: int
    
@dataclass
class Supplier:
    """Supplier data structure with performance metrics"""
    name: str
    unit_costs: Dict[str, float]  # component_name -> cost
    reliability_score: float  # 0-1 scale
    quality_score: float  # 0-1 scale
    capacity_utilization: float  # 0-1 scale
    lead_time: int  # days

@dataclass
class SourcingDecision:
    """Sourcing decision result"""
    component_name: str
    strategy: str  # 'single' or 'dual'
    allocations: Dict[str, float]  # supplier_name -> allocation_percentage
    confidence_score: float  # 0-1 scale
    processing_time: float  # seconds

In [ ]:
class PriorityBasedSourcingAlgorithm:
    """Priority-based sourcing algorithm with composite scoring"""
    
    def __init__(self, cost_weight: float = 0.4, reliability_weight: float = 0.4, 
                 capacity_weight: float = 0.2, dual_sourcing_threshold: float = 0.15):
        """
        Initialize the priority-based sourcing algorithm
        
        Args:
            cost_weight: Weight for cost efficiency in scoring (0-1)
            reliability_weight: Weight for supply reliability in scoring (0-1)
            capacity_weight: Weight for capacity considerations in scoring (0-1)
            dual_sourcing_threshold: Minimum score gap to trigger dual sourcing (0-1)
        """
        self.cost_weight = cost_weight
        self.reliability_weight = reliability_weight
        self.capacity_weight = capacity_weight
        self.dual_sourcing_threshold = dual_sourcing_threshold
        
        # Normalize weights
        total_weight = cost_weight + reliability_weight + capacity_weight
        self.cost_weight /= total_weight
        self.reliability_weight /= total_weight
        self.capacity_weight /= total_weight
    
    def calculate_cost_score(self, component: Component, supplier: Supplier) -> float:
        """
        Calculate cost efficiency score (higher is better)
        
        Cost score is inversely proportional to unit cost
        Normalized across all suppliers for this component
        """
        unit_cost = supplier.unit_costs[component.name]
        # Lower cost = higher score
        return 1.0 / (1.0 + unit_cost / 10.0)  # Normalize by typical cost scale
    
    def calculate_reliability_score(self, component: Component, supplier: Supplier) -> float:
        """
        Calculate reliability score (higher is better)
        
        Adjusted by component criticality - more critical components value reliability more
        """
        base_reliability = supplier.reliability_score
        # Criticality amplifies the importance of reliability
        criticality_factor = 1.0 + component.criticality_score * 0.5
        return base_reliability * criticality_factor
    
    def calculate_capacity_score(self, component: Component, supplier: Supplier) -> float:
        """
        Calculate capacity score (higher is better)
        
        Considers current capacity utilization and lead time
        """
        # Lower utilization = higher capacity score
        utilization_score = 1.0 - supplier.capacity_utilization
        
        # Shorter lead time = higher score
        lead_time_score = 1.0 / (1.0 + supplier.lead_time / 30.0)  # Normalize by month
        
        return (utilization_score + lead_time_score) / 2.0
    
    def calculate_composite_score(self, component: Component, supplier: Supplier) -> float:
        """
        Calculate weighted composite score for supplier-component pair
        """
        cost_score = self.calculate_cost_score(component, supplier)
        reliability_score = self.calculate_reliability_score(component, supplier)
        capacity_score = self.calculate_capacity_score(component, supplier)
        
        composite_score = (
            self.cost_weight * cost_score +
            self.reliability_weight * reliability_score +
            self.capacity_weight * capacity_score
        )
        
        return composite_score
    
    def rank_suppliers_for_component(self, component: Component, suppliers: List[Supplier]) -> List[Tuple[Supplier, float]]:
        """
        Rank suppliers for a specific component based on composite scores
        
        Returns list of (supplier, score) tuples sorted by score (descending)
        """
        supplier_scores = []
        
        for supplier in suppliers:
            score = self.calculate_composite_score(component, supplier)
            supplier_scores.append((supplier, score))
        
        # Sort by score (descending)
        supplier_scores.sort(key=lambda x: x[1], reverse=True)
        
        return supplier_scores
    
    def calculate_confidence_score(self, ranked_suppliers: List[Tuple[Supplier, float]]) -> float:
        """
        Calculate confidence score based on score separation
        
        Higher confidence when there's clear separation between top suppliers
        """
        if len(ranked_suppliers) < 2:
            return 1.0
        
        top_score = ranked_suppliers[0][1]
        second_score = ranked_suppliers[1][1]
        
        # Score gap as percentage of top score
        score_gap = (top_score - second_score) / top_score
        
        # Map score gap to confidence (0.1 to 1.0)
        confidence = min(1.0, 0.1 + score_gap * 2)
        
        return confidence
    
    def make_sourcing_decision(self, component: Component, suppliers: List[Supplier]) -> SourcingDecision:
        """
        Make sourcing decision for a single component using priority-based algorithm
        """
        start_time = time.time()
        
        # Rank suppliers for this component
        ranked_suppliers = self.rank_suppliers_for_component(component, suppliers)
        
        # Calculate confidence score
        confidence = self.calculate_confidence_score(ranked_suppliers)
        
        # Make decision based on scores and thresholds
        if len(ranked_suppliers) == 1:
            # Only one supplier available
            supplier = ranked_suppliers[0][0]
            allocations = {supplier.name: 1.0}
            strategy = 'single'
        else:
            top_supplier, top_score = ranked_suppliers[0]
            second_supplier, second_score = ranked_suppliers[1]
            
            # Check if dual sourcing is warranted
            score_gap = (top_score - second_score) / top_score
            
            if score_gap < self.dual_sourcing_threshold and component.criticality_score > 0.7:
                # High criticality + close scores = dual sourcing
                allocations = {
                    top_supplier.name: 0.7,  # 70% to top supplier
                    second_supplier.name: 0.3  # 30% to second supplier
                }
                strategy = 'dual'
            else:
                # Clear winner = single sourcing
                allocations = {top_supplier.name: 1.0}
                strategy = 'single'
        
        processing_time = time.time() - start_time
        
        return SourcingDecision(
            component_name=component.name,
            strategy=strategy,
            allocations=allocations,
            confidence_score=confidence,
            processing_time=processing_time
        )
    
    def solve_sourcing_problem(self, components: List[Component], suppliers: List[Supplier]) -> List[SourcingDecision]:
        """
        Solve the complete sourcing problem using priority-based algorithm
        
        Processes components in order of criticality (highest first)
        """
        start_time = time.time()
        
        # Sort components by criticality (descending)
        sorted_components = sorted(components, key=lambda c: c.criticality_score, reverse=True)
        
        decisions = []
        
        for component in sorted_components:
            decision = self.make_sourcing_decision(component, suppliers)
            decisions.append(decision)
        
        total_time = time.time() - start_time
        
        print(f"Priority-based algorithm completed in {total_time:.4f} seconds")
        print(f"Processed {len(components)} components with {len(suppliers)} suppliers")
        
        return decisions

In [ ]:
# Create the concrete example from the source material
def create_concrete_example():
    """Create the MediTech example with data from the source"""
    
    # Components from source: Component 1 (Critical) and Component 2 (Standard)
    components = [
        Component(
            name="Component_1_Critical",
            criticality_score=0.9,
            annual_demand=460  # Sum of [100, 120, 110, 130] from source
        ),
        Component(
            name="Component_2_Standard",
            criticality_score=0.6,
            annual_demand=810  # Sum of [200, 180, 220, 210] from source
        )
    ]
    
    # Suppliers from source with cost data and assumed performance metrics
    suppliers = [
        Supplier(
            name="Supplier_A",
            unit_costs={
                "Component_1_Critical": 15,
                "Component_2_Standard": 8
            },
            reliability_score=0.85,
            quality_score=0.87,
            capacity_utilization=0.75,
            lead_time=14
        ),
        Supplier(
            name="Supplier_B",
            unit_costs={
                "Component_1_Critical": 18,
                "Component_2_Standard": 7
            },
            reliability_score=0.92,
            quality_score=0.94,
            capacity_utilization=0.65,
            lead_time=10
        ),
        Supplier(
            name="Supplier_C",
            unit_costs={
                "Component_1_Critical": 16,
                "Component_2_Standard": 9
            },
            reliability_score=0.88,
            quality_score=0.90,
            capacity_utilization=0.80,
            lead_time=12
        )
    ]
    
    return components, suppliers

# Create the instance
components, suppliers = create_concrete_example()
print(f"Created {len(components)} components and {len(suppliers)} suppliers")
print(f"Components: {[c.name for c in components]}")
print(f"Suppliers: {[s.name for s in suppliers]}")

Created 2 components and 3 suppliers
Components: ['Component_1_Critical', 'Component_2_Standard']
Suppliers: ['Supplier_A', 'Supplier_B', 'Supplier_C']


In [ ]:
# Initialize and run the priority-based sourcing algorithm
algorithm = PriorityBasedSourcingAlgorithm(
    cost_weight=0.4,
    reliability_weight=0.4,
    capacity_weight=0.2,
    dual_sourcing_threshold=0.15
)

# Solve the sourcing problem
decisions = algorithm.solve_sourcing_problem(components, suppliers)

print("\n=== PRIORITY-BASED SOURCING RESULTS ===")
for decision in decisions:
    print(f"\n{decision.component_name}:")
    print(f"  Strategy: {decision.strategy.upper()} SOURCING")
    print(f"  Confidence: {decision.confidence_score:.1%}")
    print(f"  Processing time: {decision.processing_time:.4f} seconds")
    
    if decision.strategy == 'single':
        supplier_name = list(decision.allocations.keys())[0]
        print(f"  Allocation: 100% from {supplier_name}")
    else:
        print(f"  Allocation: Dual Sourcing")
        for supplier, allocation in decision.allocations.items():
            print(f"    {allocation*100:.0f}% from {supplier}")

Priority-based algorithm completed in 0.0000 seconds
Processed 2 components with 3 suppliers

=== PRIORITY-BASED SOURCING RESULTS ===

Component_1_Critical:
  Strategy: DUAL SOURCING
  Confidence: 17.8%
  Processing time: 0.0000 seconds
  Allocation: Dual Sourcing
    70% from Supplier_B
    30% from Supplier_C

Component_2_Standard:
  Strategy: SINGLE SOURCING
  Confidence: 25.6%
  Processing time: 0.0000 seconds
  Allocation: 100% from Supplier_B


In [ ]:
# Detailed score analysis for each component
def analyze_detailed_scores():
    """Analyze detailed scoring for transparency and understanding"""
    
    print("=== DETAILED SCORE ANALYSIS ===")
    
    for component in components:
        print(f"\n{component.name} (Criticality: {component.criticality_score:.1f}):")
        
        # Get ranked suppliers
        ranked_suppliers = algorithm.rank_suppliers_for_component(component, suppliers)
        
        print("  Supplier Rankings:")
        for i, (supplier, composite_score) in enumerate(ranked_suppliers, 1):
            cost_score = algorithm.calculate_cost_score(component, supplier)
            reliability_score = algorithm.calculate_reliability_score(component, supplier)
            capacity_score = algorithm.calculate_capacity_score(component, supplier)
            
            print(f"    {i}. {supplier.name} - Overall Score: {composite_score:.3f}")
            print(f"       Cost Score: {cost_score:.3f} (Unit Cost: ${supplier.unit_costs[component.name]})")
            print(f"       Reliability Score: {reliability_score:.3f} (Base: {supplier.reliability_score:.2f})")
            print(f"       Capacity Score: {capacity_score:.3f} (Utilization: {supplier.capacity_utilization:.1%}, Lead Time: {supplier.lead_time} days)")
        
        # Calculate confidence
        confidence = algorithm.calculate_confidence_score(ranked_suppliers)
        print(f"  Decision Confidence: {confidence:.1%}")
        
        if len(ranked_suppliers) >= 2:
            score_gap = (ranked_suppliers[0][1] - ranked_suppliers[1][1]) / ranked_suppliers[0][1]
            print(f"  Score Gap (1st vs 2nd): {score_gap:.1%}")
            
            if score_gap < algorithm.dual_sourcing_threshold and component.criticality_score > 0.7:
                print(f"  → Dual sourcing recommended (gap < {algorithm.dual_sourcing_threshold:.1%} and high criticality)")
            else:
                print(f"  → Single sourcing recommended")

analyze_detailed_scores()

=== DETAILED SCORE ANALYSIS ===

Component_1_Critical (Criticality: 0.9):
  Supplier Rankings:
    1. Supplier_B - Overall Score: 0.786
       Cost Score: 0.357 (Unit Cost: $18)
       Reliability Score: 1.334 (Base: 0.92)
       Capacity Score: 0.550 (Utilization: 65.0%, Lead Time: 10 days)
    2. Supplier_C - Overall Score: 0.756
       Cost Score: 0.385 (Unit Cost: $16)
       Reliability Score: 1.276 (Base: 0.88)
       Capacity Score: 0.457 (Utilization: 80.0%, Lead Time: 12 days)
    3. Supplier_A - Overall Score: 0.746
       Cost Score: 0.400 (Unit Cost: $15)
       Reliability Score: 1.232 (Base: 0.85)
       Capacity Score: 0.466 (Utilization: 75.0%, Lead Time: 14 days)
  Decision Confidence: 17.8%
  Score Gap (1st vs 2nd): 3.9%
  → Dual sourcing recommended (gap < 15.0% and high criticality)

Component_2_Standard (Criticality: 0.6):
  Supplier Rankings:
    1. Supplier_B - Overall Score: 0.824
       Cost Score: 0.588 (Unit Cost: $7)
       Reliability Score: 1.196 (Base: 0.

In [ ]:
# Calculate total costs and compare with baseline
def calculate_total_costs(decisions: List[SourcingDecision]) -> Dict[str, float]:
    """Calculate total annual costs for the sourcing decisions"""
    
    total_cost = 0
    cost_breakdown = {}
    
    for decision in decisions:
        component = next(c for c in components if c.name == decision.component_name)
        component_cost = 0
        
        for supplier_name, allocation in decision.allocations.items():
            supplier = next(s for s in suppliers if s.name == supplier_name)
            unit_cost = supplier.unit_costs[component.name]
            annual_cost = component.annual_demand * allocation * unit_cost
            component_cost += annual_cost
            
            # Add to total cost breakdown by supplier
            if supplier_name not in cost_breakdown:
                cost_breakdown[supplier_name] = 0
            cost_breakdown[supplier_name] += annual_cost
        
        cost_breakdown[f"{decision.component_name}_total"] = component_cost
        total_cost += component_cost
    
    cost_breakdown['grand_total'] = total_cost
    return cost_breakdown

# Calculate costs for our solution
cost_breakdown = calculate_total_costs(decisions)

print("=== COST ANALYSIS ===")
print(f"Total Annual Cost: ${cost_breakdown['grand_total']:,.2f}")
print("\nCost Breakdown by Component:")
for component in components:
    component_total = cost_breakdown.get(f"{component.name}_total", 0)
    print(f"  {component.name}: ${component_total:,.2f}")

print("\nCost Breakdown by Supplier:")
for supplier in suppliers:
    supplier_cost = cost_breakdown.get(supplier.name, 0)
    if supplier_cost > 0:
        percentage = (supplier_cost / cost_breakdown['grand_total']) * 100
        print(f"  {supplier.name}: ${supplier_cost:,.2f} ({percentage:.1f}%)")

# Compare with pure single sourcing (cheapest supplier for each component)
print("\n=== COMPARISON WITH PURE SINGLE SOURCING ===")
single_sourcing_cost = 0
for component in components:
    # Find cheapest supplier for this component
    cheapest_supplier = min(suppliers, key=lambda s: s.unit_costs[component.name])
    cheapest_cost = component.annual_demand * cheapest_supplier.unit_costs[component.name]
    single_sourcing_cost += cheapest_cost
    print(f"  {component.name}: {cheapest_supplier.name} at ${cheapest_cost:,.2f}")

print(f"\nPure Single Sourcing Total: ${single_sourcing_cost:,.2f}")
cost_difference = cost_breakdown['grand_total'] - single_sourcing_cost
print(f"Cost Premium for Priority-Based Strategy: ${cost_difference:,.2f} ({(cost_difference/single_sourcing_cost)*100:.1f}%)")

=== COST ANALYSIS ===
Total Annual Cost: $13,674.00

Cost Breakdown by Component:
  Component_1_Critical: $8,004.00
  Component_2_Standard: $5,670.00

Cost Breakdown by Supplier:
  Supplier_B: $11,466.00 (83.9%)
  Supplier_C: $2,208.00 (16.1%)

=== COMPARISON WITH PURE SINGLE SOURCING ===
  Component_1_Critical: Supplier_A at $6,900.00
  Component_2_Standard: Supplier_B at $5,670.00

Pure Single Sourcing Total: $12,570.00
Cost Premium for Priority-Based Strategy: $1,104.00 (8.8%)


In [ ]:
try:
    # Create comprehensive visualizations
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Priority-Based Sourcing Algorithm Analysis', fontsize=16, fontweight='bold')
    
    # 1. Composite Score Comparison
    ax1 = axes[0, 0]
    score_data = []
    for component in components:
        ranked_suppliers = algorithm.rank_suppliers_for_component(component, suppliers)
        for i, (supplier, score) in enumerate(ranked_suppliers):
            score_data.append({
                'Component': component.name.replace('_', ' ').title(),
                'Supplier': supplier.name.replace('Supplier_', 'S'),
                'Score': score,
                'Rank': i + 1
            })
    
    score_df = pd.DataFrame(score_data)
    sns.barplot(data=score_df, x='Component', y='Score', hue='Supplier', ax=ax1)
    ax1.set_title('Composite Scores by Component and Supplier')
    ax1.set_ylabel('Composite Score')
    ax1.legend(title='Supplier', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax1.tick_params(axis='x', rotation=45)
    
    # 2. Decision Confidence Analysis
    ax2 = axes[0, 1]
    confidence_data = [{
        'Component': d.component_name.replace('_', ' ').title(),
        'Confidence': d.confidence_score,
        'Strategy': d.strategy.title()
    } for d in decisions]
    
    confidence_df = pd.DataFrame(confidence_data)
    colors = ['lightcoral' if s == 'Single' else 'lightblue' for s in confidence_df['Strategy']]
    bars = ax2.bar(confidence_df['Component'], confidence_df['Confidence'], color=colors)
    ax2.set_title('Decision Confidence by Component')
    ax2.set_ylabel('Confidence Score')
    ax2.set_ylim(0, 1)
    ax2.tick_params(axis='x', rotation=45)
    
    # Add confidence values on bars
    for bar, confidence in zip(bars, confidence_df['Confidence']):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{confidence:.1%}', ha='center', va='bottom', fontweight='bold')
    
    # Add legend for strategy colors
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='lightcoral', label='Single Sourcing'),
                       Patch(facecolor='lightblue', label='Dual Sourcing')]
    ax2.legend(handles=legend_elements, loc='upper right')
    
    # 3. Cost Comparison
    ax3 = axes[1, 0]
    comparison_data = {
        'Priority-Based': cost_breakdown['grand_total'],
        'Pure Single Sourcing': single_sourcing_cost
    }
    
    bars = ax3.bar(comparison_data.keys(), comparison_data.values(), 
                  color=['skyblue', 'lightcoral'])
    ax3.set_title('Total Annual Cost Comparison')
    ax3.set_ylabel('Total Cost ($)')
    ax3.tick_params(axis='x', rotation=45)
    
    # Add value labels on bars
    for bar, cost in zip(bars, comparison_data.values()):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                 f'${cost:,.0f}', ha='center', va='bottom', fontweight='bold')
    
    # 4. Processing Time Performance
    ax4 = axes[1, 1]
    processing_times = [d.processing_time * 1000 for d in decisions]  # Convert to milliseconds
    component_names = [d.component_name.replace('_', ' ').title() for d in decisions]
    
    bars = ax4.bar(component_names, processing_times, color='gold', alpha=0.8)
    ax4.set_title('Algorithm Processing Time by Component')
    ax4.set_ylabel('Processing Time (milliseconds)')
    ax4.tick_params(axis='x', rotation=45)
    
    # Add time values on bars
    for bar, time_ms in zip(bars, processing_times):
        ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{time_ms:.2f}ms', ha='center', va='bottom', fontweight='bold')
    
    # Add total time annotation
    total_time = sum(processing_times)
    ax4.axhline(y=total_time/len(processing_times), color='red', linestyle='--', alpha=0.7)
    ax4.text(0.5, total_time/len(processing_times) + 0.05, 
             f'Average: {total_time/len(processing_times):.2f}ms', 
             ha='center', va='bottom', color='red', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Error in cell: {e}')

[Timeout after 120s - cell wrapped for safety]

In [ ]:
try:
    # Performance analysis and scalability test
    def scalability_test():
        """Test algorithm performance with different problem sizes"""
        
        print("=== SCALABILITY ANALYSIS ===")
        
        # Test different problem sizes
        test_sizes = [
            (2, 3),   # Current size: 2 components, 3 suppliers
            (5, 5),   # Medium: 5 components, 5 suppliers
            (10, 8),  # Large: 10 components, 8 suppliers
            (20, 10)  # Very large: 20 components, 10 suppliers
        ]
        
        performance_results = []
        
        for n_components, n_suppliers in test_sizes:
            # Generate test data
            test_components = [
                Component(
                    name=f"Component_{i}",
                    criticality_score=np.random.uniform(0.3, 0.95),
                    annual_demand=np.random.randint(100, 2000)
                )
                for i in range(n_components)
            ]
            
            test_suppliers = [
                Supplier(
                    name=f"Supplier_{j}",
                    unit_costs={f"Component_{i}": np.random.uniform(5, 25) 
                              for i in range(n_components)},
                    reliability_score=np.random.uniform(0.7, 0.95),
                    quality_score=np.random.uniform(0.8, 0.98),
                    capacity_utilization=np.random.uniform(0.4, 0.9),
                    lead_time=np.random.randint(5, 30)
                )
                for j in range(n_suppliers)
            ]
            
            # Run algorithm and measure performance
            start_time = time.time()
            test_decisions = algorithm.solve_sourcing_problem(test_components, test_suppliers)
            end_time = time.time()
            
            processing_time = end_time - start_time
            dual_sourcing_count = sum(1 for d in test_decisions if d.strategy == 'dual')
            avg_confidence = np.mean([d.confidence_score for d in test_decisions])
            
            performance_results.append({
                'Components': n_components,
                'Suppliers': n_suppliers,
                'Processing_Time_s': processing_time,
                'Processing_Time_ms': processing_time * 1000,
                'Dual_Sourcing_Count': dual_sourcing_count,
                'Dual_Sourcing_Percentage': (dual_sourcing_count / n_components) * 100,
                'Avg_Confidence': avg_confidence
            })
        
        # Display results
        performance_df = pd.DataFrame(performance_results)
        print(performance_df.to_string(index=False))
        
        return performance_df
    
    # Run scalability test
    performance_df = scalability_test()
except Exception as e:
    print(f'Error in cell: {e}')

[Timeout after 120s - cell wrapped for safety]

### Why this Tier exists vs earlier Tiers
This Tier 2 provides a practical heuristic approach that addresses the limitations of mathematical optimization:
- **Computational efficiency**: Processes decisions in milliseconds vs. hours for exact methods
- **Scalability**: Handles large problem sizes that are intractable for mathematical optimization
- **Practical decision rules**: Clear, interpretable decision logic for real-world procurement
- **Confidence metrics**: Provides decision quality indicators for risk management
- **Sequential processing**: Enables real-time decision making in dynamic environments

### Pros / Cons vs earlier Tiers
**Advantages over Tier 1 (Mathematical Optimization):**
- **Speed**: 100-1000x faster computation time
- **Scalability**: Handles 10-100x larger problem instances
- **Simplicity**: Easier to implement and understand
- **Flexibility**: Can incorporate qualitative factors easily
- **Real-time capability**: Suitable for dynamic decision environments

**Disadvantages vs Tier 1:**
- **Optimality**: No guarantee of optimal solutions
- **Systematic exploration**: Limited search of solution space
- **Precision**: Approximate solutions with potential suboptimality gaps
- **Parameter sensitivity**: Performance depends on weight and threshold settings

### When to use this Tier
- **Large-scale problems**: When mathematical optimization becomes computationally intractable
- **Real-time decisions**: When sourcing decisions must be made quickly
- **Limited data**: When precise parameter estimates are unavailable
- **Practical procurement**: When decision rules need to be transparent and explainable
- **Dynamic environments**: When conditions change frequently and rapid re-optimization is needed
- **Initial screening**: As a fast method to identify promising sourcing strategies before detailed optimization